In [ ]:
! uv pip install langchain openai tiktoken rapidocr python-dotenv langchain-community

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("Set GOOGLE_API_KEY in your .env file")

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

In [ ]:
##data ingestion
import torch
print(torch.cuda.get_device_name(0))

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('../rich.txt')
documents = loader.load()

In [ ]:
documents[0].page_content[:500]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
chunks = splitter.split_documents(documents)

In [ ]:
print(chunks[0].page_content)

In [ ]:
! uv pip install faiss-cpu langchain-google-genai

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001",
    google_api_key = GEMINI_API_KEY
)

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
query = "What is the main topic of the document?"
docs = vectorstore.similarity_search(query)

for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
        You are an assistant for question-answering tasks.
        Use the following pieces of retrieved context to answer the question.
        Use ten sentences maximum and keep the answer concise.
        Question: {question}
        Context: {context}
        Answer:
    """

prompt = PromptTemplate.from_template(template)

In [ ]:
prompt

In [ ]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
llm.invoke("hello")

In [ ]:
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever()
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} | 
    prompt | 
    llm | 
    output_parser
    )

In [ ]:
rag_chain.invoke("What is the author?")nm